# Silent To Speech - Backend Server

This notebook runs the Heavy AI Models (Whisper, T5, MediaPipe, Edge-TTS) on Google Colab's free cloud GPU. 
It creates a FastAPI server and exposes it to the public internet using LocalTunnel so your local frontend can connect to it.

In [ ]:
!pip install fastapi uvicorn python-multipart torch torchaudio transformers openai-whisper mediapipe==0.10.14 ffmpeg-python noisereduce edge-tts scikit-learn nest-asyncio
!npm install -g localtunnel

In [ ]:
%%writefile audio_pipeline.py
import os
import subprocess
import whisper
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import asyncio
import edge_tts
import torch
import librosa
import soundfile as sf
import noisereduce as nr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper model on {device}...")
whisper_model = whisper.load_model("base", device=device)

print(f"Loading T5 model on {device}...")
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small", legacy=False)
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

def enhance_audio(input_path, temp_clean_path):
    norm_path = input_path + "_norm.wav"
    command = ["ffmpeg", "-y", "-i", input_path, "-filter:a", "loudnorm", norm_path]
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        y, sr = librosa.load(norm_path, sr=None)
        reduced_noise = nr.reduce_noise(y=y, sr=sr)
        sf.write(temp_clean_path, reduced_noise, sr)
    except:
        import shutil
        shutil.copy(norm_path, temp_clean_path)
    if os.path.exists(norm_path): os.remove(norm_path)

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path)
    return result["text"]

def clean_text(text):
    if not text.strip(): return ""
    inputs = t5_tokenizer(f"Fix grammar and remove stutters from the following text: {text}", return_tensors="pt").to(device)
    outputs = t5_model.generate(**inputs, max_length=100)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

async def text_to_speech(text, output_path):
    voice = "en-US-AriaNeural"
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)

def process_audio(input_path, output_path):
    temp_clean_path = input_path + "_temp.wav"
    try:
        enhance_audio(input_path, temp_clean_path)
        raw_text = transcribe_audio(temp_clean_path)
        cleaned_text = clean_text(raw_text)
        asyncio.run(text_to_speech(cleaned_text, output_path))
        return cleaned_text
    finally:
        if os.path.exists(temp_clean_path): os.remove(temp_clean_path)


In [ ]:
%%writefile video_pipeline.py
import cv2
import mediapipe as mp
import numpy as np
import asyncio
from sklearn.neighbors import KNeighborsClassifier
from audio_pipeline import text_to_speech

mp_holistic = mp.solutions.holistic
dummy_X = np.random.rand(10, 21 * 3)
dummy_y = ["Hello", "How", "Are", "You", "Thank", "You", "Good", "Morning", "Yes", "No"]
knn_classifier = KNeighborsClassifier(n_neighbors=1)
knn_classifier.fit(dummy_X, dummy_y)

def extract_landmarks(frame, holistic_model):
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = holistic_model.process(image)
    if results.right_hand_landmarks:
        return np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten()
    return None

def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    detected_words = []
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            if frame_count % 10 == 0:
                landmarks = extract_landmarks(frame, holistic)
                if landmarks is not None:
                    prediction = knn_classifier.predict([landmarks])[0]
                    if not detected_words or detected_words[-1] != prediction:
                        detected_words.append(prediction)
            frame_count += 1
    cap.release()
    transcript = " ".join(detected_words) if detected_words else "No sign language detected."
    asyncio.run(text_to_speech(transcript, output_path))
    return transcript


In [ ]:
import os
import threading
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uuid
import shutil
from audio_pipeline import process_audio
from video_pipeline import process_video

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

os.makedirs("uploads", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

@app.post("/api/process-audio")
async def handle_audio(file: UploadFile = File(...)):
    file_ext = os.path.splitext(file.filename)[1]
    file_id = str(uuid.uuid4())
    input_path = os.path.join("uploads", f"{file_id}{file_ext}")
    with open(input_path, "wb") as buffer: shutil.copyfileobj(file.file, buffer)
    output_path = os.path.join("outputs", f"{file_id}_clean.wav")
    transcript = process_audio(input_path, output_path)
    return JSONResponse({"transcript": transcript, "audio_url": f"/outputs/{file_id}_clean.wav"})

@app.post("/api/process-video")
async def handle_video(file: UploadFile = File(...)):
    file_ext = os.path.splitext(file.filename)[1]
    file_id = str(uuid.uuid4())
    input_path = os.path.join("uploads", f"{file_id}{file_ext}")
    with open(input_path, "wb") as buffer: shutil.copyfileobj(file.file, buffer)
    output_path = os.path.join("outputs", f"{file_id}_sign.wav")
    transcript = process_video(input_path, output_path)
    return JSONResponse({"transcript": transcript, "audio_url": f"/outputs/{file_id}_sign.wav"})

from fastapi.staticfiles import StaticFiles
app.mount("/outputs", StaticFiles(directory="outputs"), name="outputs")

def run_server():
    import uvicorn
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

print("\n=== SERVER IS RUNNING ===")
print("Generating Public URL...")
!lt --port 8000